# 04 - Multi-Market Fetch + Backtest

目标：拉取指定股票（美股/港股/沪深/新加坡）真实数据，统一写入 ClickHouse，并做策略回测与买卖点标记。

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 兼容从项目根目录或 notebooks 目录启动 Jupyter 的两种场景
cwd = Path.cwd()
candidates = [cwd / 'src', cwd.parent / 'src']
for p in candidates:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from quant_research.db import get_client, insert_df, query_df
from quant_research.market_data import fetch_one
from quant_research.strategies import (
    list_strategies,
    run_strategy_backtest,
    run_enhanced_breakout_portfolio_backtest,
)

sns.set_theme(style='whitegrid')

In [ ]:
# 1) 配置股票与时间
targets = [
    {'market': 'US', 'code': 'AAPL'},
    {'market': 'HK', 'code': '0700'},
    {'market': 'CN', 'code': '600519'},
    {'market': 'SG', 'code': 'D05'},
]

start_date = '2023-01-01'
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')

In [ ]:
# 2) 拉取并汇总（fetch_one 已兼容 yfinance MultiIndex）
frames = []
for t in targets:
    part = fetch_one(t['market'], t['code'], start=start_date, end=end_date)
    part['market'] = t['market'].upper()
    part['code'] = t['code'].upper()
    part['yf_ticker'] = part['symbol'].str.split(':').str[-1]
    frames.append(part)

all_bars = pd.concat(frames, ignore_index=True)
all_bars = all_bars.sort_values(['symbol', 'trade_date']).drop_duplicates(['symbol', 'trade_date']).reset_index(drop=True)
all_bars.head()

In [ ]:
# 3) 建表 + 清理重跑重复区间 + 入库
client = get_client()

client.command('''
CREATE TABLE IF NOT EXISTS market_bars
(
    trade_date Date,
    symbol LowCardinality(String),
    market LowCardinality(String),
    code String,
    yf_ticker String,
    open Float64,
    high Float64,
    low Float64,
    close Float64,
    volume Float64,
    amount Float64,
    vwap Float64,
    ret_1d Float64,
    ret_5d Float64,
    created_at DateTime DEFAULT now()
)
ENGINE = MergeTree
PARTITION BY toYYYYMM(trade_date)
ORDER BY (symbol, trade_date)
''')

client.command('''
CREATE TABLE IF NOT EXISTS strategy_signals
(
    trade_date Date,
    symbol LowCardinality(String),
    strategy_name LowCardinality(String),
    signal_type LowCardinality(String),
    price Float64,
    position Int8,
    created_at DateTime DEFAULT now()
)
ENGINE = MergeTree
PARTITION BY toYYYYMM(trade_date)
ORDER BY (strategy_name, symbol, trade_date)
''')

symbols = sorted(all_bars['symbol'].unique().tolist())
min_date = str(pd.to_datetime(all_bars['trade_date']).dt.date.min())
max_date = str(pd.to_datetime(all_bars['trade_date']).dt.date.max())
symbols_csv = ', '.join(["'" + s.replace("'", "''") + "'" for s in symbols])

client.command(f"""
ALTER TABLE market_bars
DELETE WHERE symbol IN ({symbols_csv})
  AND trade_date BETWEEN toDate('{min_date}') AND toDate('{max_date}')
""")

client.command(f"""
ALTER TABLE daily_bars
DELETE WHERE symbol IN ({symbols_csv})
  AND trade_date BETWEEN toDate('{min_date}') AND toDate('{max_date}')
""")

client.close()

market_bars_for_insert = all_bars[['trade_date', 'symbol', 'market', 'code', 'yf_ticker', 'open', 'high', 'low', 'close', 'volume', 'amount', 'vwap', 'ret_1d', 'ret_5d']].copy()
insert_df('market_bars', market_bars_for_insert)

daily_bars_for_insert = all_bars[['trade_date', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'amount', 'vwap', 'ret_1d', 'ret_5d']].copy()
insert_df('daily_bars', daily_bars_for_insert)

print('inserted market_bars:', len(market_bars_for_insert))
print('inserted daily_bars:', len(daily_bars_for_insert))

In [ ]:
# 4) 策略选择与回测（单票 / 组合双模式）
list_strategies()

# 模式: 'single_symbol' 或 'portfolio'
strategy_mode = 'portfolio'

# 单票模式可选策略:
# - dual_ma
# - sma_regression_filter
# - rsi_mean_reversion
# - bollinger_breakout
strategy_name = 'sma_regression_filter'

# 单票参数（会覆盖默认参数）
strategy_params = {
    'short_win': 10,
    'long_win': 30,
    'reg_win': 20,
    'total_capital': 1_000_000.0,
}

# 组合策略参数（对应你提供的优化版逻辑）
portfolio_params = {
    'initial_capital': 1_000_000.0,
    'top_n': 10,
    'rebalance_days': 20,
    'cooldown_days': 10,
    'min_trade_ratio': 0.02,
    'breakout_filter': 1.01,
    'slippage': 0.001,
    'fee_rate': 0.0003,
    'lot_size': 100,
    'log_level': 'INFO',
}

if strategy_mode == 'single_symbol':
    symbol_to_test = all_bars['symbol'].iloc[0]
    symbol_df = all_bars[all_bars['symbol'] == symbol_to_test].copy()
    bt, metrics = run_strategy_backtest(symbol_df, strategy_name=strategy_name, params=strategy_params)
    trades = bt[bt['signal_type'].isin(['BUY', 'SELL'])][['trade_date', 'signal_type', 'close', 'position']].copy()
    trades['symbol'] = symbol_to_test
    trades = trades.rename(columns={'signal_type': 'side', 'close': 'price', 'position': 'shares'})
    logs = pd.DataFrame()
    result_title = f"Single Symbol: {symbol_to_test} ({strategy_name})"
    metrics_df = pd.DataFrame([{'symbol': symbol_to_test, **metrics}])
else:
    bt, trades, metrics = run_enhanced_breakout_portfolio_backtest(all_bars, params=portfolio_params)
    logs = bt.attrs.get('logs', pd.DataFrame())
    result_title = 'Portfolio: enhanced_breakout_portfolio'
    metrics_df = pd.DataFrame([metrics])

metrics_df

In [ ]:
# 5) 可视化 + 买卖点 + 收益率曲线
fig, axes = plt.subplots(3, 1, figsize=(14, 14), sharex=True)

if strategy_mode == 'single_symbol':
    buy = bt[bt['signal_type'] == 'BUY']
    sell = bt[bt['signal_type'] == 'SELL']
    axes[0].plot(bt['trade_date'], bt['close'], label='Close', color='steelblue')
    if 'sma_short' in bt.columns:
        axes[0].plot(bt['trade_date'], bt['sma_short'], label='SMA Short', color='orange', alpha=0.9)
    if 'sma_long' in bt.columns:
        axes[0].plot(bt['trade_date'], bt['sma_long'], label='SMA Long', color='green', alpha=0.9)
    axes[0].scatter(buy['trade_date'], buy['close'], marker='^', s=70, color='red', label='BUY')
    axes[0].scatter(sell['trade_date'], sell['close'], marker='v', s=70, color='black', label='SELL')
else:
    axes[0].plot(bt['trade_date'], bt['equity'], label='Portfolio Equity', color='steelblue')
    if len(trades) > 0:
        buy = trades[trades['side'] == 'BUY']
        sell = trades[trades['side'] == 'SELL']
        buy_y = bt.set_index('trade_date').reindex(pd.to_datetime(buy['trade_date']))['equity'].values
        sell_y = bt.set_index('trade_date').reindex(pd.to_datetime(sell['trade_date']))['equity'].values
        axes[0].scatter(pd.to_datetime(buy['trade_date']), buy_y, marker='^', s=40, color='red', label='BUY')
        axes[0].scatter(pd.to_datetime(sell['trade_date']), sell_y, marker='v', s=40, color='black', label='SELL')

axes[0].set_title(result_title)
axes[0].legend()

axes[1].plot(bt['trade_date'], bt['benchmark_curve'], label='Benchmark Capital Curve', color='gray')
if strategy_mode == 'single_symbol':
    axes[1].plot(bt['trade_date'], bt['strategy_curve'], label='Strategy Capital Curve', color='purple')
else:
    axes[1].plot(bt['trade_date'], bt['equity'], label='Strategy Capital Curve', color='purple')
axes[1].set_title('Capital Curve')
axes[1].legend()

axes[2].plot(bt['trade_date'], bt['benchmark_cum_return'], label='Benchmark Cumulative Return', color='gray')
if strategy_mode == 'single_symbol':
    axes[2].plot(bt['trade_date'], bt['strategy_cum_return'], label='Strategy Cumulative Return', color='purple')
else:
    axes[2].plot(bt['trade_date'], bt['strategy_cum_return'], label='Strategy Cumulative Return', color='purple')
axes[2].axhline(0, color='black', linewidth=1, alpha=0.4)
axes[2].set_title('Return Curve (Cumulative Return)')
axes[2].set_ylabel('Return')
axes[2].legend()

plt.xticks(rotation=30)
plt.tight_layout()

In [ ]:
# 6) 保存买卖点到 ClickHouse（覆盖同区间）
if strategy_mode == 'single_symbol':
    signals_to_save = bt[bt['signal_type'].isin(['BUY', 'SELL'])][['trade_date', 'close', 'position', 'signal_type']].copy()
    signals_to_save = signals_to_save.rename(columns={'close': 'price'})
    signals_to_save['symbol'] = symbol_to_test
    signals_to_save['strategy_name'] = strategy_name
    signals_to_save = signals_to_save[['trade_date', 'symbol', 'strategy_name', 'signal_type', 'price', 'position']]
else:
    if len(trades) > 0:
        signals_to_save = trades[['trade_date', 'symbol', 'side', 'price', 'shares']].copy()
        signals_to_save = signals_to_save.rename(columns={'side': 'signal_type'})
        signals_to_save['position'] = 0
        signals_to_save['strategy_name'] = 'enhanced_breakout_portfolio'
        signals_to_save = signals_to_save[['trade_date', 'symbol', 'strategy_name', 'signal_type', 'price', 'position']]
    else:
        signals_to_save = pd.DataFrame(columns=['trade_date','symbol','strategy_name','signal_type','price','position'])

if not signals_to_save.empty:
    s_min = str(pd.to_datetime(signals_to_save['trade_date']).dt.date.min())
    s_max = str(pd.to_datetime(signals_to_save['trade_date']).dt.date.max())
    strategy_to_save = signals_to_save['strategy_name'].iloc[0]

    client = get_client()
    symbols = sorted(signals_to_save['symbol'].unique().tolist())
    symbols_csv = ', '.join(["'" + s.replace("'", "''") + "'" for s in symbols])
    client.command(f"""
    ALTER TABLE strategy_signals
    DELETE WHERE strategy_name = '{strategy_to_save}'
      AND symbol IN ({symbols_csv})
      AND trade_date BETWEEN toDate('{s_min}') AND toDate('{s_max}')
    """)
    client.close()

    insert_df('strategy_signals', signals_to_save)

print('inserted signals:', len(signals_to_save))

if not signals_to_save.empty:
    strategy_to_save = signals_to_save['strategy_name'].iloc[0]
    query_df(f"""
    select trade_date, symbol, strategy_name, signal_type, price, position
    from strategy_signals
    where strategy_name = '{strategy_to_save}'
    order by trade_date desc
    limit 20
    """)
else:
    pd.DataFrame()